In [ ]:
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point

# Path to the folder containing the CSV files
folder_path = './bikeshare_data/nyc_bikeshare'
# Use glob to find all CSV files in the folder
csv_files = glob.glob(f"{folder_path}/*.csv")

df = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)
df.columns

In [ ]:
import matplotlib.pyplot as plt

# 1. Aggregate station data: median lat/lng, first name
start_station_info = (
    df.groupby('start_station_id')
      .agg({
          'start_lat': 'median',
          'start_lng': 'median',
          'start_station_name': 'first'
      })
      .reset_index()
      .rename(columns={
          'start_station_id': 'Station ID',
          'start_lat': 'Station Latitude',
          'start_lng': 'Station Longitude',
          'start_station_name': 'Station Name'
      })
)

end_station_info = (
    df.groupby('end_station_id')
      .agg({
          'end_lat': 'median',
          'end_lng': 'median',
          'end_station_name': 'first'
      })
      .reset_index()
      .rename(columns={
          'end_station_id': 'Station ID',
          'end_lat': 'Station Latitude',
          'end_lng': 'Station Longitude',
          'end_station_name': 'Station Name'
      })
)

# 2. Combine, deduplicate by Station ID
all_stations = pd.concat([start_station_info, end_station_info])
all_stations_cleaned = all_stations.drop_duplicates(subset=['Station ID'])

# 3. Filter out stations with invalid or out-of-bounds coordinates
all_stations_cleaned = all_stations_cleaned[
    (all_stations_cleaned['Station Latitude'].between(40, 41)) &
    (all_stations_cleaned['Station Longitude'].between(-75, -73))
]

# 4. Create geometry and load shapefiles
geometry = [Point(xy) for xy in zip(all_stations_cleaned['Station Longitude'],
                                    all_stations_cleaned['Station Latitude'])]
stations_gdf = gpd.GeoDataFrame(all_stations_cleaned, geometry=geometry)

ny_tracts = gpd.read_file('./tract_shapefiles/tl_2023_ny_tract/tl_2023_36_tract.shp')
nj_tracts = gpd.read_file('./tract_shapefiles/tl_2023_nj_tract/tl_2023_34_tract.shp')
tracts = pd.concat([ny_tracts, nj_tracts], ignore_index=True)
stations_gdf = stations_gdf.set_crs(ny_tracts.crs)

# 5. Spatial join stations → tracts
stations_with_tracts = gpd.sjoin(stations_gdf, tracts, how="left", predicate="within")
stations_with_tracts.rename(columns={'Station ID': 'id'}, inplace=True)

# 6. Calculate trips per station
trips_out = df.groupby('start_station_id').size().reset_index(name='trips_out')
trips_in = df.groupby('end_station_id').size().reset_index(name='trips_in')
total_trips = pd.merge(trips_out, trips_in, left_on='start_station_id', right_on='end_station_id', how='outer')
total_trips['total_trips'] = total_trips['trips_out'].fillna(0) + total_trips['trips_in'].fillna(0)
total_trips.rename(columns={'start_station_id': 'id'}, inplace=True)
total_trips = total_trips.drop(columns=['end_station_id', 'trips_in', 'trips_out'])

# 7. Merge trips with station/tract GeoDataFrame
stations_with_tracts_and_trips = pd.merge(
    stations_with_tracts,
    total_trips,
    on='id',
    how='left'
)

# 8. Aggregate total trips by tract
tracts_with_trips = stations_with_tracts_and_trips.groupby('GEOID', as_index=False).agg({
    'total_trips': 'sum',
    'ALAND': 'first'
})

# 9. Normalize by number of days in study window
tracts_with_trips['trips_per_day'] = tracts_with_trips['total_trips'] / 821

# 10. Save final result
tracts_with_trips.to_csv('./bikeshare_data/tracts_with_trips/nyc_tracts_with_trips.csv', index=False)

# Print column names for verification
print(tracts_with_trips.columns)


In [ ]:
stations_with_tracts_and_trips[stations_with_tracts_and_trips['Station Latitude'] < 40]

In [ ]:
nyc_csv_files = ['./nyc_agg_tract/transport_main.csv', 
                './nyc_agg_tract/selected_econ_main.csv', 
                './nyc_agg_tract/geo_mobility_main.csv',
                './nyc_agg_tract/nyc_commute_by_sex_s0801.csv',
                './total_pop_s0601.csv',
                './full_s0101.csv'
                ]

variable_meanings = {}

merged_nyc_data = pd.read_csv(nyc_csv_files[0])
merged_nyc_data['GEOID'] = merged_nyc_data['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
merged_nyc_data = merged_nyc_data.drop(columns=['GEO_ID'])  # Drop redundant column

meanings = merged_nyc_data.iloc[0].to_dict()  # Convert the first row to a dictionary
variable_meanings.update(meanings)  # Update the global dictionary of meanings
merged_nyc_data = merged_nyc_data.iloc[1:].reset_index(drop=True)

# Sequentially merge the remaining CSVs
for file in nyc_csv_files[1:]:
    df = pd.read_csv(file)
    df['GEOID'] = df['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
    df = df.drop(columns=['GEO_ID'])  # Drop redundant column
    df = df.drop(columns=['NAME'])
    meanings = df.iloc[0].to_dict()  # Convert the first row to a dictionary
    variable_meanings.update(meanings)  # Update the global dictionary of meanings
    df = df.iloc[1:].reset_index(drop=True)

    merged_nyc_data = pd.merge(merged_nyc_data, df, on='GEOID', how='outer')  # Merge on GEOID


# Merge with tracts_with_trips DataFrame
merged_df = pd.merge(tracts_with_trips, merged_nyc_data, on='GEOID', how='left')

# Ensure all object columns are converted to numeric where possible
for col in merged_df.select_dtypes(include=['object']).columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')


# Exclude columns whose names end with M
columns_to_exclude = [col for col in merged_df.columns if col.endswith('M')]
filtered_columns = merged_df.columns.difference(columns_to_exclude)

# Select numeric columns that are not excluded
numeric_columns = merged_df[filtered_columns].select_dtypes(include=['number']).columns
merged_df = merged_df[numeric_columns]

merged_df['trips_per_day'] = merged_df['total_trips'] / 548

# List of columns to transform
columns_to_transform = [col for col in merged_df.columns if col not in ['GEOID', 'ALAND', 'total_trips', 'trips_per_day']]

# Creating new columns by dividing by population and area
for col in columns_to_transform:
    merged_df[f'{col}_per_capita'] = merged_df[col] / merged_df['S0101_C01_001E'] # total population
    merged_df[f'{col}_per_ALAND'] = merged_df[col] / merged_df['ALAND']

# Save combined dataframe for Multiple Regression analysis
merged_df.to_csv('./multiple_regression/nyc_census_data_and_trips.csv')

# Calculate correlation between bike trips and all other numeric columns
correlations = merged_df.corrwith(merged_df['total_trips'])
#correlations = merged_df[numeric_columns].corrwith(merged_df['total_trips'] / merged_df['ALAND'])

# Convert to a DataFrame for better readability and sort by absolute correlation
correlation_df = correlations.reset_index()
correlation_df.columns = ['Variable', 'Correlation']
correlation_df['Absolute_Correlation'] = correlation_df['Correlation'].abs()

# Sort by absolute correlation in descending order
sorted_correlation_df = correlation_df.sort_values(by='Absolute_Correlation', ascending=False)

# Extracting the relevant prefix for variable names
def extract_prefix(variable_name):
    for i, char in enumerate(variable_name):
        if char.upper() == 'E':
            return variable_name[:i + 1]
    return variable_name  # Return the whole variable name if 'E' is not found

# Apply the function to create a new column 'Meaning'
sorted_correlation_df['Meaning'] = sorted_correlation_df['Variable'].apply(lambda x: variable_meanings.get(extract_prefix(x), 'Unknown'))

# Display the top variables
columns_to_print = ['Meaning', 'Correlation']
print(sorted_correlation_df[columns_to_print])


In [ ]:
output_path = './correlation_results/nyc_correlations.csv'
sorted_correlation_df.to_csv(output_path, index=False)